In [ ]:
from dotenv import load_dotenv

# Don't need to specify full path if notbook file and .env file are on the same leve in the project structure
load_dotenv("../.env")

True

In [79]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, CharacterTextSplitter
from langchain_aws import BedrockEmbeddings, ChatBedrockConverse
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
import numpy as np
from langchain_core.runnables import RunnablePassthrough

In [7]:
loader_pdf = PyPDFLoader("365_Data_Science_Courses.pdf")

In [10]:
pages_pdf = loader_pdf.load()
len(pages_pdf)

3

In [15]:
pages_string = ""
for page in pages_pdf:
    pages_string += page.page_content

In [16]:
pages_string

'# Introduction to Data and Data Science \n## Big Data Real-life Examples \n \nAll right! So… Where do we find big data? The answer is in increasingly more industries and \ncompanies. Here are a few notable examples. \nAs one of the largest online communities, ‘Facebook’ keeps track of its user’s names, personal \ndata, photos, videos, recorded messages and so on. This means their data has a lot of variety. And \nwith over 2 billion users worldwide, the volume of data stored on their servers is tremendous. \nFacebook requires real-time reporting of the aggregated, anonymized voice of its users, and it \napplies many analytical tools for its mobile applications. This means the company is investing in \nboosting its real-time data processing powers, increasing the velocity of its data set. \nGreat stuff! \nNow, where else can we find ‘big data’? \nLet’s take financial trading data for example. What happens when we record the stock price every 5 \nseconds? Or every single second? We get a

In [17]:
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "Course Title"),("##", "Lecture Title")]
)

In [21]:
pages_md_split = md_splitter.split_text(pages_string)
len(pages_md_split)

4

In [24]:
char_text_splitter = CharacterTextSplitter(
    separator=".",
    chunk_size=400,
    chunk_overlap=0
)

In [42]:
pages_char_splitter = char_text_splitter.split_documents(pages_md_split)

In [43]:
len(pages_char_splitter)

17

In [54]:
embedding = BedrockEmbeddings(model_id="amazon.titan-embed-text-v1", region_name="us-east-1", normalize=True)

In [55]:
v1 = embedding.embed_query(pages_char_splitter[1].page_content)


In [56]:
v2 = embedding.embed_query(pages_char_splitter[8].page_content)

In [60]:
cosine_similarity = np.dot(v1, v2) 
cosine_similarity

np.float64(0.3031850991109374)

In [62]:
vectorstore = Chroma.from_documents(persist_directory="./365-ds-courses",
                                embedding=embedding,
                                documents=pages_char_splitter)

In [63]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.7},
)

In [64]:
question = "Could you give me some examples of Big Data used in real life?"

In [68]:
retrieved_docs = retriever.invoke(question)

In [69]:
for i in retrieved_docs:
    print(f"Page Content: {i.page_content}\n----------\n")

Page Content: We hope these examples clarify the use of big data in the real world.
Catch you later!
----------

Page Content: Great stuff!
Now, where else can we find ‘big data’?
Let’s take financial trading data for example. What happens when we record the stock price every 5
seconds? Or every single second? We get a dataset that is incredibly voluminous, requiring
significantly more memory, disk space and various techniques to extract meaningful information
from it. Data like this would be also considered ‘big data’
----------

Page Content: All right! So… Where do we find big data? The answer is in increasingly more industries and
companies. Here are a few notable examples.
As one of the largest online communities, ‘Facebook’ keeps track of its user’s names, personal
data, photos, videos, recorded messages and so on. This means their data has a lot of variety
----------



In [88]:
TEMPLATE_1 = '''
You are a helpful Q&A chatbot at the 365 Data Science online educational platform. 
The platform offers video courses on data science topics. 
Your task is to provide students with exhaustive answers to their questions.

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture and the course this context is taken from in the format:
Resources: 
1. Lecture "Lecture Title" from the "Course Title" course
…
where "Lecture Title" and "Course Title" should be substituted with the respective titles of all resource lectures.
'''

TEMPLATE_2 = """{question}"""

In [89]:
message_template_1 = SystemMessagePromptTemplate.from_template(TEMPLATE_1)
message_template_2 = HumanMessagePromptTemplate.from_template(TEMPLATE_2)

In [90]:
chat_template = ChatPromptTemplate.from_messages([
    message_template_1,
    message_template_2
])

In [91]:
chat = ChatBedrockConverse(
    model="us.anthropic.claude-sonnet-4-6",
    region_name="us-east-1",
    max_tokens=250,
    temperature=0
)

In [92]:
str_parser = StrOutputParser()

In [93]:
chain = {
    "context": retriever,
    "question": RunnablePassthrough(),
} | chat_template | chat | str_parser

In [94]:
response = chain.invoke(question)

In [95]:
print(response)

# Real-Life Examples of Big Data

Sure! Here are some notable examples of Big Data used in real life:

## 1. 📱 Social Media – Facebook
Facebook, as one of the largest online communities, keeps track of enormous amounts of user information, including:
- Names and personal data
- Photos and videos
- Recorded messages

This data has a **lot of variety**, making it a prime example of Big Data.

## 2. 📈 Financial Trading Data
In financial markets, stock prices can be recorded **every 5 seconds or even every single second**. This generates a dataset that is:
- **Incredibly voluminous**
- Requires significantly more **memory and disk space**
- Demands various specialized **techniques to extract meaningful information**

This type of high-frequency data is also considered Big Data.

---

These examples highlight how Big Data appears across **increasingly more industries and companies**, from social media platforms to financial markets. The key characteristics that make data "big" include its *